In [2]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix,accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV
from sklearn.feature_selection import SelectKBest, f_classif,chi2
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from lightgbm import LGBMClassifier

df = pd.read_csv('./train1.csv')
X_train = df.drop(['target','id'], axis=1)


In [3]:
y_train = df['target']

In [4]:
X_train,X_test,y_train,y_test = train_test_split(X_train, y_train, test_size=0.3, random_state=29173, stratify=y_train)

In [5]:
X_test.shape

(88863, 65)

In [6]:
"""df_test = pd.read_csv('test.csv')
X_test = df_test.drop('id',axis=1) 
test_ids = df_test['id']"""

"df_test = pd.read_csv('test.csv')\nX_test = df_test.drop('id',axis=1) \ntest_ids = df_test['id']"

In [7]:
categorical_cols = [col for col in X_train.columns if col.endswith('_cat')]
X_train[categorical_cols] = X_train[categorical_cols].astype('category')
X_test[categorical_cols] = X_test[categorical_cols].astype('category')
binary_cols = [col for col in X_train.columns if col.endswith('_bin')]
X_train[binary_cols] = X_train[binary_cols].astype(bool)
X_test[binary_cols] = X_test[binary_cols].astype(bool)

In [8]:
def simple_imputer(X_train, X_val, categorical_columns=None, numerical_columns=None):
    """
    Impute missing values in train and validation/test sets using only training data statistics.
    - categorical_columns: list of categorical columns to impute with mode
    - numerical_columns: list of numerical columns to impute with mean
    """
    X_train_imputed = X_train.copy()
    X_val_imputed = X_val.copy()

    # Categorical columns
    if categorical_columns:
        for col in categorical_columns:
            if col in X_train_imputed.columns:
                mode_value = X_train_imputed[col].mode(dropna=True)
                if not mode_value.empty:
                    mode_value = mode_value[0]
                    X_train_imputed[col].fillna(mode_value, inplace=True)
                    if col in X_val_imputed.columns:
                        X_val_imputed[col].fillna(mode_value, inplace=True)

    # Numerical columns
    if numerical_columns:
        for col in numerical_columns:
            if col in X_train_imputed.columns:
                mean_value = X_train_imputed[col].mean(skipna=True)
                if pd.notna(mean_value):  # ensure mean is valid
                    X_train_imputed[col].fillna(mean_value, inplace=True)
                    if col in X_val_imputed.columns:
                        X_val_imputed[col].fillna(mean_value, inplace=True)

    return X_train_imputed, X_val_imputed


In [9]:
num_cols = [col for col in X_train.columns if not col.endswith(('_cat', '_bin'))]

In [10]:
X_train_imputed,X_test_imputed = simple_imputer(X_train, X_test, categorical_cols,num_cols)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22780\1639349455.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_train_imputed[col].fillna(mode_value, inplace=True)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22780\1639349455.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

In [11]:
X_train_imputed['ps_car_12'].isnull().sum()

np.int64(0)

In [12]:
for col in X_train_imputed.select_dtypes(['category']).columns:
    X_train_imputed[col] = X_train_imputed[col].cat.as_ordered()
    X_train_imputed[col] = X_train_imputed[col].cat.codes.astype('int')
    X_test_imputed[col] = X_test_imputed[col].cat.as_ordered()
    X_test_imputed[col] = X_test_imputed[col].cat.codes.astype('int')


In [13]:
selected_features_anova_chi = ['ps_car_13', 'ps_reg_02', 'ps_car_12', 'feature4', 'ps_reg_03', 'feature2', 'ps_car_15', 'ps_ind_15', 'ps_reg_01', 'ps_ind_01', 'feature5', 'ps_car_14', 'feature7', 'ps_ind_03', 'ps_calc_01', 'ps_car_04_cat', 'ps_ind_05_cat', 'ps_car_11_cat', 'ps_car_06_cat', 'ps_car_01_cat', 'ps_car_02_cat', 'ps_ind_04_cat', 'ps_car_08_cat', 'ps_car_09_cat', 'ps_car_05_cat', 'ps_ind_17_bin', 'ps_ind_07_bin', 'ps_ind_06_bin', 'ps_ind_16_bin', 'ps_ind_08_bin', 'ps_ind_09_bin', 'ps_ind_12_bin']

In [15]:
cat_features = X_train[selected_features_anova_chi].select_dtypes(include=['category']).columns.tolist()

In [14]:
# Split a small validation set from training data
lgbm_clf = LGBMClassifier(
    boosting_type='gbdt',
    objective='binary',          # or 'multiclass' for multi-class problems
    metric='auc',                # or 'multi_logloss' for multiclass
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,                # -1 means no limit
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# Fit with eval_set
lgbm_clf.fit(
    X_train_imputed[selected_features_anova_chi], 
    y_train,
)

# Evaluate
y_pred_proba = lgbm_clf.predict_proba(X_test_imputed[selected_features_anova_chi])[:, 1]
roc_auc = roc_auc_score(y_test, y_pred_proba)
print("Baseline AUROC:", roc_auc)


[LightGBM] [Info] Number of positive: 10630, number of negative: 196716
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009695 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1927
[LightGBM] [Info] Number of data points in the train set: 207346, number of used features: 32
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.051267 -> initscore=-2.918081
[LightGBM] [Info] Start training from score -2.918081
Baseline AUROC: 0.628510533844423


In [15]:
val_auroc = roc_auc_score(y_test, y_pred_proba)
print(f"Validation AUROC: {val_auroc:.4f}")

Validation AUROC: 0.6285


In [ ]:
param_grid = {
    'num_leaves': [50,63, 80],
    'max_depth': [4, 5, 6,7],
    'learning_rate': [0.01, 0.02,.03,.04],
    'n_estimators': [1000,1100,1200],
    'subsample': [0.75, 0.8, 0.85,.9],
    'colsample_bytree': [.5,.6,.55],
    'reg_alpha': [0.4, 0.5, 0.55],     # L1 regularization
    'reg_lambda': [.4, 0.5, 0.55],    # L2 regularization
    'min_child_samples': [25, 30, 35]
}
'''best Parameters: {'subsample': 0.8, 'reg_lambda': 0.5, 'reg_alpha': 0.5, 'num_leaves': 63, 
                  'n_estimators': 1000, 'min_child_samples': 30, 'max_depth': 5, 
                  'learning_rate': 0.01, 'colsample_bytree': 0.6}'''



In [17]:
random_search = RandomizedSearchCV(
    estimator=lgbm_clf,          # or lgbm_reg
    param_distributions=param_grid,
    n_iter=30,                   # number of parameter combinations to try
    scoring='roc_auc',           # or 'neg_mean_squared_error' for regression
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_imputed[selected_features_anova_chi], y_train)

print("Best Parameters:", random_search.best_params_)
print("Best Score:", random_search.best_score_)

y_pred_proba = random_search.best_estimator_.predict_proba(
    X_test_imputed[selected_features_anova_chi]
)[:, 1]

roc_auc = roc_auc_score(y_test, y_pred_proba)
print("Test AUROC:", roc_auc)


Fitting 5 folds for each of 30 candidates, totalling 150 fits
[LightGBM] [Info] Number of positive: 10630, number of negative: 196716
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.046076 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1927
[LightGBM] [Info] Number of data points in the train set: 207346, number of used features: 32
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.051267 -> initscore=-2.918081
[LightGBM] [Info] Start training from score -2.918081
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

In [19]:
param_grid = {
    'iterations': [400,500,600],
    'depth': [3, 4, 5],
    'learning_rate': [0.03, 0.05, 0.07],
    'l2_leaf_reg': [5,6,7],
    'border_count': [50, 64 , 70],
    'bagging_temperature': [0, 0.05, 0.1],
    'random_strength': [0.9, 1,1.2],
    'grow_policy': ['SymmetricTree', 'Depthwise', 'Lossguide']
}
'''Best Parameters: {'random_strength': 1, 'learning_rate': 0.05, 'l2_leaf_reg': 5, 'iterations': 500, 
                  'grow_policy': 'Depthwise', 'depth': 4, 'border_count': 64, 'bagging_temperature': 0}'''

X_train_final, X_valid, y_train_final, y_valid = train_test_split(
    X_train_imputed[selected_features_anova_chi],
    y_train,
    test_size=0.3,
    random_state=29173
)
# Base CatBoost model
catboost_model = CatBoostClassifier(
    verbose=0,
    random_state=29173,
    eval_metric='AUC'
)


# RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=catboost_model,
    param_distributions=param_grid,
    n_iter=30,
    scoring='roc_auc',
    cv=5,
    verbose=2,
    n_jobs=-1
)

random_search.fit(
    X_train_final,
    y_train_final,
    cat_features=cat_features,
    eval_set=(X_valid, y_valid)
)

print("Best Parameters:", random_search.best_params_)

y_pred_proba = random_search.best_estimator_.predict_proba(
    X_test_imputed[selected_features_anova_chi]
)[:, 1]

roc_auc = roc_auc_score(y_test, y_pred_proba)
print("Test AUROC:", roc_auc)


Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters: {'random_strength': 1, 'learning_rate': 0.03, 'l2_leaf_reg': 5, 'iterations': 500, 'grow_policy': 'Lossguide', 'depth': 5, 'border_count': 70, 'bagging_temperature': 0.1}
Test AUROC: 0.637711594065058


In [ ]:
param_grid_new = {
    'n_estimators': [180, 200, 210],
    'learning_rate': [0.03, 0.05, 0.06],
    'max_depth': [3, 5, 7, 10],
    'min_child_weight': [4,5,6],
    'gamma': [0.05, 0.1,.13, 0.15],
    'subsample': [0.75, 0.8, .85],
    'colsample_bytree': [0.5,.6,.65],
    'reg_alpha': [0.4, 0.5,.6],
    'reg_lambda': [.7,.9,1,1.1],
}
"""Best Parameters: {'subsample': 0.85, 'reg_lambda': 1.1, 'reg_alpha': 0.6, 'n_estimators': 200, 
                  'min_child_weight': 6, 'max_depth': 3, 'learning_rate': 0.06, 
                  'gamma': 0.05, 'colsample_bytree': 0.6}"""

In [ ]:
xx

In [ ]:
xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

xgb_random_search_new = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid_new,
    n_iter=50,                 # number of random combinations
    scoring='roc_auc',         # or 'accuracy', 'f1', etc.
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

xgb_random_search_new.fit(X_train_imputed[selected_features_anova_chi], y_train)

print("Best Parameters:", xgb_random_search_new.best_params_)
print("Best Score:", xgb_random_search_new.best_score_)
best_xg_new = xgb_random_search_new.best_estimator_
y_val_pred_proba_new = best_xg_new.predict_proba(X_test_imputed)[:, 1]
val_auroc_new = roc_auc_score(y_test, y_val_pred_proba_new)
print(f"Validation AUROC: {val_auroc_new:.4f}")

In [ ]:
xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

xgb_random_search_new = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid_new,
    n_iter=50,                 # number of random combinations
    scoring='roc_auc',         # or 'accuracy', 'f1', etc.
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

xgb_random_search_new.fit(X_train_imputed, y_train)

print("Best Parameters:", xgb_random_search_new.best_params_)
print("Best Score:", xgb_random_search_new.best_score_)
best_xg_new = xgb_random_search_new.best_estimator_


In [ ]:
param_grid_updated_selected = {
    'n_estimators': [190, 200, 205],
    'learning_rate': [0.055, 0.06,.065],
    'max_depth': [1,2,3,4],
    'min_child_weight': [6.5,6,7],
    'gamma': [0.04,.05,.055],
    'subsample': [.85,.9,.95],
    'colsample_bytree': [0.55,.6,.63],
    'reg_alpha': [.6,.65,.7],
    'reg_lambda': [1.1,1.3,1.5]
}


In [ ]:
xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

xgb_random_search_updated_selected = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid_updated_selected,
    n_iter=50,                 # number of random combinations
    scoring='roc_auc',         # or 'accuracy', 'f1', etc.
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

xgb_random_search_updated_selected.fit(X_train_imputed[selected_features_anova_chi], y_train)

print("Best Parameters:", xgb_random_search_updated_selected.best_params_)
print("Best Score:", xgb_random_search_updated_selected.best_score_)


In [ ]:
best_xg_updated = xgb_random_search_updated_selected.best_estimator_
y_val_pred_proba_updated = best_xg_updated.predict_proba(X_test_imputed[selected_features_anova_chi])[:, 1]
val_auroc_updated = roc_auc_score(y_test, y_val_pred_proba_updated)
print(f"Validation AUROC: {val_auroc_updated:.4f}")